# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_16 — Deep Hedging Intro with PyTorch

### Research question

Can a neural network learn a discrete-time hedging policy for an option when trading is costly and continuous Black-Scholes delta hedging is no longer directly optimal?

This notebook connects Phase 2 option pricing with Phase 3 statistical learning:

```text
02_03_black_scholes_merton_pde
02_06_monte_carlo_gbm_and_fat_tails
02_08_dynamic_delta_hedging_simulation
02_15_greeks_pathwise_vs_likelihood_ratio
03_07_lstm_forecaster
03_15_purged_feature_selection_for_time_series
```

The purpose is not to claim that neural networks magically hedge better.

The purpose is to introduce the **deep hedging** idea:

> Instead of deriving a closed-form hedge rule, define a hedging objective and train a neural network policy directly on simulated market paths.

It covers:

1. European call liability;
2. GBM path simulation;
3. Black-Scholes price and delta baselines;
4. discrete-time hedging PnL;
5. transaction costs;
6. neural hedge policy in PyTorch;
7. differentiable hedging loss;
8. mean-variance and CVaR-style risk objectives;
9. training loop;
10. out-of-sample test paths;
11. no hedge versus BSM delta versus neural hedge;
12. stress testing under wrong volatility and jumps;
13. hedge turnover and cost diagnostics;
14. limitations and research extensions.

The key idea:

> Deep hedging is risk-control learning, not return prediction. The model learns how to trade a hedge book under frictions and a chosen loss function.

## 1. Classical delta hedging baseline

Under Black-Scholes assumptions, a European call price is:

$$
C(S,t)=S N(d_1)-Ke^{-r\tau}N(d_2)
$$

where:

$$
d_1 = \frac{ \log(S/K)+(r+\frac{1}{2}\sigma^2)\tau }{ \sigma\sqrt{\tau} }
$$

$$
d_2=d_1-\sigma\sqrt{\tau}
$$

The delta is:

$$
\Delta = N(d_1)
$$

In continuous time with no transaction costs and correct model assumptions, delta hedging replicates the option.

But real hedging is:

- discrete;
- costly;
- constrained;
- model-misspecified;
- exposed to jumps and liquidity effects.

That is where learned hedging policies become interesting.

## 2. Deep hedging formulation

We sell an option and hedge with the underlying.

At time step $t$, the policy chooses a hedge position:

$$
h_t = \pi_\theta(X_t)
$$

where $X_t$ may include:

- time to maturity;
- log-moneyness;
- current underlying price;
- previous hedge position;
- volatility estimate;
- spread/cost state;
- option Greeks or other features.

Terminal hedging PnL:

$$
\begin{aligned}
PnL_T &= premium \\
&\quad - payoff(S_T) \\
&\quad + \sum_{t=0}^{N-1} h_t(S_{t+1}-S_t) \\
&\quad - \sum_{t=0}^{N-1} cost_t
\end{aligned}
$$

The training objective can be:

### Mean squared hedging error

$$
\mathcal L = \mathbb E[(-PnL_T)^2]
$$

### Mean-variance risk

$$
\mathcal L = -\mathbb E[PnL_T]+\lambda Var(PnL_T)
$$

### CVaR-style tail risk

$$
\mathcal L = -\mathbb E[PnL_T] + \lambda CVaR_\alpha(-PnL_T)
$$

This notebook uses simple differentiable objectives suitable for a first PyTorch implementation.

## 3. What this notebook does not claim

This notebook does **not** claim:

1. neural hedging always beats Black-Scholes delta;
2. simulated performance transfers directly to real markets;
3. a network trained under GBM is robust to all market conditions;
4. deep hedging is alpha;
5. the model discovers arbitrage.

It is an introductory infrastructure notebook.

The honest interpretation is:

> Given a simulator, a cost model, and a loss function, a neural network can learn a hedging policy. Whether that policy is useful depends on validation, stress testing, and realism of the simulator.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

TORCH_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class DeepHedgingConfig:
    S0: float = 100.0
    K: float = 100.0
    r: float = 0.00
    sigma_train: float = 0.20
    sigma_test: float = 0.20
    T: float = 1.0
    n_steps: int = 30
    n_train_paths: int = 16_384
    n_valid_paths: int = 4_096
    n_test_paths: int = 8_192
    transaction_cost_rate: float = 0.0010
    max_abs_hedge: float = 2.0
    learning_rate: float = 0.001
    batch_size: int = 1024
    epochs: int = 30
    risk_aversion: float = 2.0
    cvar_alpha: float = 0.95
    seed: int = 42


config = DeepHedgingConfig()
config

## 5. Black-Scholes helper functions

We implement:

1. normal CDF;
2. Black-Scholes call price;
3. Black-Scholes delta;
4. call payoff.

These are used as baselines.

In [ ]:
def normal_cdf(x):
    x = np.asarray(x, dtype=float)
    erf_vec = np.vectorize(math.erf)
    return 0.5 * (1.0 + erf_vec(x / np.sqrt(2.0)))


def bsm_call_price(S, K, r, sigma, tau):
    S = np.asarray(S, dtype=float)
    tau = np.asarray(tau, dtype=float)

    tau_safe = np.maximum(tau, 1e-12)
    sigma_safe = max(sigma, 1e-12)

    d1 = (np.log(np.maximum(S, 1e-12) / K) + (r + 0.5 * sigma_safe**2) * tau_safe) / (sigma_safe * np.sqrt(tau_safe))
    d2 = d1 - sigma_safe * np.sqrt(tau_safe)

    price = S * normal_cdf(d1) - K * np.exp(-r * tau_safe) * normal_cdf(d2)

    intrinsic = np.maximum(S - K, 0.0)

    return np.where(tau <= 1e-12, intrinsic, price)


def bsm_call_delta(S, K, r, sigma, tau):
    S = np.asarray(S, dtype=float)
    tau = np.asarray(tau, dtype=float)

    tau_safe = np.maximum(tau, 1e-12)
    sigma_safe = max(sigma, 1e-12)

    d1 = (np.log(np.maximum(S, 1e-12) / K) + (r + 0.5 * sigma_safe**2) * tau_safe) / (sigma_safe * np.sqrt(tau_safe))

    delta = normal_cdf(d1)
    terminal_delta = (S > K).astype(float)

    return np.where(tau <= 1e-12, terminal_delta, delta)


def call_payoff(S, K):
    return np.maximum(S - K, 0.0)


initial_call_price = float(bsm_call_price(config.S0, config.K, config.r, config.sigma_train, config.T))
initial_call_delta = float(bsm_call_delta(config.S0, config.K, config.r, config.sigma_train, config.T))

pd.Series({
    "initial_call_price": initial_call_price,
    "initial_call_delta": initial_call_delta
})

## 6. GBM simulator

Risk-neutral GBM:

$$
dS_t = rS_tdt+\sigma S_tdW_t
$$

Exact discrete simulation:

$$
S_{t+\Delta t} = S_t\exp\Big[ (r-\frac12\sigma^2)\Delta t+\sigma\sqrt{\Delta t}Z_t \Big]
$$

This simulator is deliberately simple.

Later cells stress-test the policy under changed volatility and jumps.

In [ ]:
def simulate_gbm_paths(
    n_paths,
    n_steps,
    S0,
    r,
    sigma,
    T,
    seed=42
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps

    Z = rng.standard_normal((n_paths, n_steps))
    log_returns = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = np.log(S0)
    log_paths[:, 1:] = np.log(S0) + np.cumsum(log_returns, axis=1)

    S = np.exp(log_paths)

    return S


train_paths = simulate_gbm_paths(
    config.n_train_paths,
    config.n_steps,
    config.S0,
    config.r,
    config.sigma_train,
    config.T,
    seed=config.seed
)

valid_paths = simulate_gbm_paths(
    config.n_valid_paths,
    config.n_steps,
    config.S0,
    config.r,
    config.sigma_train,
    config.T,
    seed=config.seed + 1
)

test_paths = simulate_gbm_paths(
    config.n_test_paths,
    config.n_steps,
    config.S0,
    config.r,
    config.sigma_test,
    config.T,
    seed=config.seed + 2
)

train_paths.shape, valid_paths.shape, test_paths.shape

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(50):
    plt.plot(np.linspace(0, config.T, config.n_steps + 1), train_paths[i], alpha=0.35)
plt.title("Sample GBM Training Paths")
plt.xlabel("Time")
plt.ylabel("Underlying price")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(train_paths[:, -1], bins=80, density=True)
plt.title("Terminal Underlying Distribution")
plt.xlabel("S_T")
plt.ylabel("Density")
plt.show()

## 7. Hedging PnL engine in NumPy

For a hedge path $h_t$:

$$
\begin{aligned}
PnL_T &= premium \\
&\quad - payoff \\
&\quad + \sum_t h_t(S_{t+1}-S_t) \\
&\quad - cost
\end{aligned}
$$

Transaction cost:

$$
cost = c\sum_t |h_t-h_{t-1}|S_t
$$

where $h_{-1}=0$.

This function evaluates any hedge matrix.

In [ ]:
def evaluate_hedge_numpy(paths, hedge_positions, premium, K, cost_rate):
    S = np.asarray(paths, dtype=float)
    h = np.asarray(hedge_positions, dtype=float)

    dS = S[:, 1:] - S[:, :-1]

    hedge_pnl = np.sum(h * dS, axis=1)
    payoff = call_payoff(S[:, -1], K)

    previous_h = np.concatenate([np.zeros((len(S), 1)), h[:, :-1]], axis=1)
    trade_size = np.abs(h - previous_h)
    transaction_cost = cost_rate * np.sum(trade_size * S[:, :-1], axis=1)

    pnl = premium - payoff + hedge_pnl - transaction_cost

    return {
        "pnl": pnl,
        "hedge_pnl": hedge_pnl,
        "payoff": payoff,
        "transaction_cost": transaction_cost,
        "turnover": np.sum(trade_size, axis=1)
    }


def no_hedge_positions(paths):
    return np.zeros((paths.shape[0], paths.shape[1] - 1))


def bsm_delta_positions(paths, K, r, sigma, T):
    n_paths, n_cols = paths.shape
    n_steps = n_cols - 1
    times = np.linspace(0, T, n_cols)
    tau = T - times[:-1]

    h = np.zeros((n_paths, n_steps))

    for t in range(n_steps):
        h[:, t] = bsm_call_delta(paths[:, t], K, r, sigma, tau[t])

    return h


premium = initial_call_price

no_hedge_eval = evaluate_hedge_numpy(
    test_paths,
    no_hedge_positions(test_paths),
    premium,
    config.K,
    config.transaction_cost_rate
)

delta_hedge_positions = bsm_delta_positions(
    test_paths,
    config.K,
    config.r,
    config.sigma_train,
    config.T
)

delta_eval = evaluate_hedge_numpy(
    test_paths,
    delta_hedge_positions,
    premium,
    config.K,
    config.transaction_cost_rate
)

pd.Series({
    "no_hedge_mean_pnl": no_hedge_eval["pnl"].mean(),
    "delta_hedge_mean_pnl": delta_eval["pnl"].mean(),
    "no_hedge_pnl_std": no_hedge_eval["pnl"].std(ddof=1),
    "delta_hedge_pnl_std": delta_eval["pnl"].std(ddof=1),
    "delta_mean_cost": delta_eval["transaction_cost"].mean()
})

## 8. Risk metrics

We evaluate hedging distributions using:

- mean PnL;
- standard deviation;
- 5% quantile;
- expected shortfall of losses;
- mean transaction cost;
- mean turnover.

Define loss:

$$
L=-PnL
$$

For confidence level $\alpha$:

$$
CVaR_\alpha(L)=E[L \mid L \ge VaR_\alpha(L)]
$$

In [ ]:
def pnl_risk_metrics(pnl, transaction_cost=None, turnover=None, alpha=0.95):
    pnl = np.asarray(pnl, dtype=float)
    loss = -pnl
    var = np.quantile(loss, alpha)
    cvar = loss[loss >= var].mean() if np.any(loss >= var) else np.nan

    out = {
        "mean_pnl": float(np.mean(pnl)),
        "std_pnl": float(np.std(pnl, ddof=1)),
        "p05_pnl": float(np.quantile(pnl, 0.05)),
        "p01_pnl": float(np.quantile(pnl, 0.01)),
        "loss_var_95": float(var),
        "loss_cvar_95": float(cvar),
        "prob_loss": float(np.mean(pnl < 0))
    }

    if transaction_cost is not None:
        out["mean_transaction_cost"] = float(np.mean(transaction_cost))
        out["p95_transaction_cost"] = float(np.quantile(transaction_cost, 0.95))

    if turnover is not None:
        out["mean_turnover"] = float(np.mean(turnover))
        out["p95_turnover"] = float(np.quantile(turnover, 0.95))

    return out


baseline_metrics = pd.DataFrame([
    {"strategy": "no_hedge", **pnl_risk_metrics(no_hedge_eval["pnl"], no_hedge_eval["transaction_cost"], no_hedge_eval["turnover"])},
    {"strategy": "bsm_delta", **pnl_risk_metrics(delta_eval["pnl"], delta_eval["transaction_cost"], delta_eval["turnover"])}
])

baseline_metrics

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(no_hedge_eval["pnl"], bins=80, alpha=0.6, density=True, label="No hedge")
plt.hist(delta_eval["pnl"], bins=80, alpha=0.6, density=True, label="BSM delta hedge")
plt.axvline(0, linestyle="--")
plt.title("Terminal PnL Distribution")
plt.xlabel("PnL")
plt.ylabel("Density")
plt.legend()
plt.show()

## 9. PyTorch deep hedging model

If PyTorch is available, we train a neural policy.

Policy input at each step:

$$
X_t = [ t/T,\, \log(S_t/K),\, S_t/S_0,\, h_{t-1} ]
$$

Policy output:

$$
h_t = max\_hedge \cdot \tanh(f_\theta(X_t))
$$

The $\tanh$ keeps hedge positions bounded.

If PyTorch is unavailable, the notebook still runs all classical baselines and records that neural training was skipped.

In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(config.seed)

    class HedgePolicy(nn.Module):
        def __init__(self, input_dim=4, hidden_dim=32, max_abs_hedge=2.0):
            super().__init__()
            self.max_abs_hedge = max_abs_hedge
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, 1)
            )

        def forward(self, x):
            return self.max_abs_hedge * torch.tanh(self.net(x)).squeeze(-1)

    policy = HedgePolicy(max_abs_hedge=config.max_abs_hedge)
    policy
else:
    policy = None
    print("PyTorch unavailable: neural hedge training will be skipped.")

## 10. Differentiable hedging PnL in PyTorch

The full hedging simulation is differentiable.

At every time step:

1. compute state features;
2. neural network outputs hedge $h_t$;
3. update trading costs;
4. accumulate hedge PnL.

Terminal PnL:

$$
\begin{aligned}
PnL_T &= premium \\
&\quad - payoff \\
&\quad + hedge\_pnl \\
&\quad - transaction\_cost
\end{aligned}
$$

In [ ]:
if TORCH_AVAILABLE:
    def torch_hedging_pnl(paths_tensor, policy, config, premium):
        S = paths_tensor
        n_paths, n_cols = S.shape
        n_steps = n_cols - 1

        hedge_pnl = torch.zeros(n_paths, device=S.device)
        transaction_cost = torch.zeros(n_paths, device=S.device)
        prev_h = torch.zeros(n_paths, device=S.device)

        for t in range(n_steps):
            tau_fraction = torch.full((n_paths,), t / n_steps, device=S.device)
            log_moneyness = torch.log(torch.clamp(S[:, t] / config.K, min=1e-8))
            spot_scaled = S[:, t] / config.S0

            features = torch.stack([
                tau_fraction,
                log_moneyness,
                spot_scaled,
                prev_h
            ], dim=1)

            h = policy(features)

            dS = S[:, t + 1] - S[:, t]
            hedge_pnl = hedge_pnl + h * dS

            trade_size = torch.abs(h - prev_h)
            transaction_cost = transaction_cost + config.transaction_cost_rate * trade_size * S[:, t]

            prev_h = h

        payoff = torch.clamp(S[:, -1] - config.K, min=0.0)
        pnl = premium - payoff + hedge_pnl - transaction_cost

        return pnl, transaction_cost
else:
    print("Skipping torch_hedging_pnl definition because PyTorch unavailable.")

## 11. Deep hedging loss functions

We implement three possible losses.

### Mean squared hedging loss

$$
\mathcal L_{MSE}=E[(-PnL)^2]
$$

### Mean-variance loss

$$
\mathcal L_{MV}=-E[PnL]+\lambda Var(PnL)
$$

### CVaR-style loss

$$
\mathcal L_{CVaR}=-E[PnL]+\lambda CVaR_\alpha(-PnL)
$$

The notebook trains with a mean-variance objective by default because it is stable and simple.

In [ ]:
if TORCH_AVAILABLE:
    def deep_hedging_loss(pnl, risk_aversion=2.0, mode="mean_variance", alpha=0.95):
        if mode == "mse":
            return torch.mean((-pnl) ** 2)

        if mode == "mean_variance":
            return -torch.mean(pnl) + risk_aversion * torch.var(pnl, unbiased=False)

        if mode == "cvar":
            loss = -pnl
            var = torch.quantile(loss, alpha)
            cvar = loss[loss >= var].mean()
            return -torch.mean(pnl) + risk_aversion * cvar

        raise ValueError("mode must be mse, mean_variance, or cvar.")
else:
    print("Skipping torch loss definition because PyTorch unavailable.")

## 12. Training loop

Training uses simulated GBM paths.

Validation uses independently simulated paths.

The model is selected by validation loss.

Important:

> The network is trained on a simulator. If the simulator is wrong, the learned hedge can be wrong.

In [ ]:
def make_torch_tensor(paths):
    if not TORCH_AVAILABLE:
        return None
    return torch.tensor(paths, dtype=torch.float32)


training_history = pd.DataFrame()
best_policy_state = None

if TORCH_AVAILABLE:
    train_tensor = make_torch_tensor(train_paths)
    valid_tensor = make_torch_tensor(valid_paths)

    optimizer = torch.optim.Adam(policy.parameters(), lr=config.learning_rate)

    n_train = train_tensor.shape[0]
    indices = np.arange(n_train)

    history_rows = []
    best_valid_loss = np.inf

    for epoch in range(config.epochs):
        np.random.shuffle(indices)
        batch_losses = []

        policy.train()

        for start in range(0, n_train, config.batch_size):
            batch_idx = indices[start:start + config.batch_size]
            batch = train_tensor[batch_idx]

            pnl, tc = torch_hedging_pnl(batch, policy, config, premium)
            loss = deep_hedging_loss(
                pnl,
                risk_aversion=config.risk_aversion,
                mode="mean_variance",
                alpha=config.cvar_alpha
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(float(loss.detach().cpu()))

        policy.eval()
        with torch.no_grad():
            valid_pnl, valid_tc = torch_hedging_pnl(valid_tensor, policy, config, premium)
            valid_loss = deep_hedging_loss(
                valid_pnl,
                risk_aversion=config.risk_aversion,
                mode="mean_variance",
                alpha=config.cvar_alpha
            )

        row = {
            "epoch": epoch + 1,
            "train_loss": float(np.mean(batch_losses)),
            "valid_loss": float(valid_loss.detach().cpu()),
            "valid_mean_pnl": float(valid_pnl.mean().detach().cpu()),
            "valid_std_pnl": float(valid_pnl.std(unbiased=True).detach().cpu()),
            "valid_mean_transaction_cost": float(valid_tc.mean().detach().cpu())
        }
        history_rows.append(row)

        if row["valid_loss"] < best_valid_loss:
            best_valid_loss = row["valid_loss"]
            best_policy_state = {k: v.detach().cpu().clone() for k, v in policy.state_dict().items()}

    if best_policy_state is not None:
        policy.load_state_dict(best_policy_state)

    training_history = pd.DataFrame(history_rows)

training_history.tail() if len(training_history) else pd.DataFrame([{"note": "PyTorch unavailable; training skipped"}])

In [ ]:
if len(training_history):
    plt.figure(figsize=(10, 6))
    plt.plot(training_history["epoch"], training_history["train_loss"], label="train")
    plt.plot(training_history["epoch"], training_history["valid_loss"], label="validation")
    plt.title("Deep Hedging Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
else:
    print("No training history to plot.")

## 13. Extract neural hedge positions

We evaluate the trained neural policy out of sample.

The policy chooses hedge positions sequentially using only current state and previous hedge position.

In [ ]:
def neural_hedge_positions_numpy(paths, policy, config):
    if not TORCH_AVAILABLE or policy is None:
        return np.full((paths.shape[0], paths.shape[1] - 1), np.nan)

    policy.eval()
    S = torch.tensor(paths, dtype=torch.float32)
    n_paths, n_cols = S.shape
    n_steps = n_cols - 1

    positions = []
    prev_h = torch.zeros(n_paths)

    with torch.no_grad():
        for t in range(n_steps):
            tau_fraction = torch.full((n_paths,), t / n_steps)
            log_moneyness = torch.log(torch.clamp(S[:, t] / config.K, min=1e-8))
            spot_scaled = S[:, t] / config.S0

            features = torch.stack([
                tau_fraction,
                log_moneyness,
                spot_scaled,
                prev_h
            ], dim=1)

            h = policy(features)
            positions.append(h.numpy())
            prev_h = h

    return np.stack(positions, axis=1)


neural_positions = neural_hedge_positions_numpy(test_paths, policy, config)

if np.isfinite(neural_positions).all():
    neural_eval = evaluate_hedge_numpy(
        test_paths,
        neural_positions,
        premium,
        config.K,
        config.transaction_cost_rate
    )
else:
    neural_eval = {
        "pnl": np.full(config.n_test_paths, np.nan),
        "hedge_pnl": np.full(config.n_test_paths, np.nan),
        "payoff": call_payoff(test_paths[:, -1], config.K),
        "transaction_cost": np.full(config.n_test_paths, np.nan),
        "turnover": np.full(config.n_test_paths, np.nan)
    }

np.nanmean(neural_eval["pnl"])

## 14. Out-of-sample strategy comparison

We compare:

1. no hedge;
2. Black-Scholes delta hedge;
3. neural hedge if PyTorch is available.

Metrics focus on distribution of terminal PnL and cost/turnover.

In [ ]:
strategy_metrics = pd.DataFrame([
    {"strategy": "no_hedge", **pnl_risk_metrics(no_hedge_eval["pnl"], no_hedge_eval["transaction_cost"], no_hedge_eval["turnover"])},
    {"strategy": "bsm_delta", **pnl_risk_metrics(delta_eval["pnl"], delta_eval["transaction_cost"], delta_eval["turnover"])},
    {"strategy": "deep_hedge_pytorch", **pnl_risk_metrics(neural_eval["pnl"], neural_eval["transaction_cost"], neural_eval["turnover"])}
])

strategy_metrics

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(no_hedge_eval["pnl"], bins=80, alpha=0.45, density=True, label="No hedge")
plt.hist(delta_eval["pnl"], bins=80, alpha=0.45, density=True, label="BSM delta")
if np.isfinite(neural_eval["pnl"]).any():
    plt.hist(neural_eval["pnl"], bins=80, alpha=0.45, density=True, label="Deep hedge")
plt.axvline(0, linestyle="--")
plt.title("Test PnL Distribution")
plt.xlabel("PnL")
plt.ylabel("Density")
plt.legend()
plt.show()

## 15. Hedge policy visualisation

We inspect average hedge position as a function of time.

For Black-Scholes delta, positions are analytical.

For the neural hedge, positions are learned.

A cost-aware neural hedge may trade less aggressively than pure delta hedging.

In [ ]:
time_grid = np.linspace(0, config.T, config.n_steps)

avg_delta_position = np.mean(delta_hedge_positions, axis=0)
avg_neural_position = np.nanmean(neural_positions, axis=0)

plt.figure(figsize=(12, 6))
plt.plot(time_grid, avg_delta_position, label="Average BSM delta")
if np.isfinite(avg_neural_position).any():
    plt.plot(time_grid, avg_neural_position, label="Average neural hedge")
plt.title("Average Hedge Position Through Time")
plt.xlabel("Time")
plt.ylabel("Hedge position")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(time_grid, np.mean(np.abs(np.diff(np.c_[np.zeros(len(delta_hedge_positions)), delta_hedge_positions], axis=1)), axis=0), label="BSM delta trade size")
if np.isfinite(neural_positions).all():
    plt.plot(time_grid, np.mean(np.abs(np.diff(np.c_[np.zeros(len(neural_positions)), neural_positions], axis=1)), axis=0), label="Neural hedge trade size")
plt.title("Average Absolute Trade Size")
plt.xlabel("Time")
plt.ylabel("Average |Δ hedge|")
plt.legend()
plt.show()

## 16. Hedge position versus moneyness

We compare hedge positions against log-moneyness at selected time steps.

$$
m_t=\log(S_t/K)
$$

A call hedge generally increases with moneyness.

Transaction costs may make the neural hedge smoother or less reactive.

In [ ]:
selected_steps = [0, config.n_steps // 3, 2 * config.n_steps // 3, config.n_steps - 1]

for step in selected_steps:
    moneyness = np.log(test_paths[:, step] / config.K)
    plt.figure(figsize=(10, 6))
    plt.scatter(moneyness, delta_hedge_positions[:, step], alpha=0.15, label="BSM delta")
    if np.isfinite(neural_positions).all():
        plt.scatter(moneyness, neural_positions[:, step], alpha=0.15, label="Neural hedge")
    plt.title(f"Hedge Position vs Log-Moneyness, Step {step}")
    plt.xlabel("log(S/K)")
    plt.ylabel("Hedge position")
    plt.legend()
    plt.show()

## 17. Stress test: volatility mismatch

A learned policy trained at one volatility may fail when realised volatility differs.

We test out of sample under:

- lower volatility;
- training volatility;
- higher volatility.

This is model risk.

In [ ]:
stress_vols = [0.12, 0.20, 0.35]
stress_rows = []

for sigma in stress_vols:
    stress_paths = simulate_gbm_paths(
        config.n_test_paths,
        config.n_steps,
        config.S0,
        config.r,
        sigma,
        config.T,
        seed=config.seed + int(1000 * sigma)
    )

    delta_pos = bsm_delta_positions(stress_paths, config.K, config.r, config.sigma_train, config.T)
    delta_out = evaluate_hedge_numpy(stress_paths, delta_pos, premium, config.K, config.transaction_cost_rate)

    neural_pos = neural_hedge_positions_numpy(stress_paths, policy, config)
    if np.isfinite(neural_pos).all():
        neural_out = evaluate_hedge_numpy(stress_paths, neural_pos, premium, config.K, config.transaction_cost_rate)
    else:
        neural_out = {
            "pnl": np.full(config.n_test_paths, np.nan),
            "transaction_cost": np.full(config.n_test_paths, np.nan),
            "turnover": np.full(config.n_test_paths, np.nan)
        }

    stress_rows.append({"sigma_realized": sigma, "strategy": "bsm_delta", **pnl_risk_metrics(delta_out["pnl"], delta_out["transaction_cost"], delta_out["turnover"])})
    stress_rows.append({"sigma_realized": sigma, "strategy": "deep_hedge_pytorch", **pnl_risk_metrics(neural_out["pnl"], neural_out["transaction_cost"], neural_out["turnover"])})

vol_stress_report = pd.DataFrame(stress_rows)

vol_stress_report

In [ ]:
plt.figure(figsize=(10, 6))
for strategy, g in vol_stress_report.groupby("strategy"):
    plt.plot(g["sigma_realized"], g["loss_cvar_95"], marker="o", label=strategy)
plt.title("Volatility Stress: 95% Loss CVaR")
plt.xlabel("Realised volatility")
plt.ylabel("Loss CVaR 95%")
plt.legend()
plt.show()

## 18. Stress test: jump diffusion

Black-Scholes delta hedging assumes continuous paths.

Jumps create unhedgeable gap risk.

We simulate jump-diffusion paths and evaluate the same policies.

In [ ]:
def simulate_jump_diffusion_paths(
    n_paths,
    n_steps,
    S0,
    r,
    sigma,
    T,
    jump_intensity,
    jump_mean,
    jump_vol,
    seed=123
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps

    Z = rng.standard_normal((n_paths, n_steps))
    N = rng.poisson(jump_intensity * dt, size=(n_paths, n_steps))
    J = rng.normal(jump_mean, jump_vol, size=(n_paths, n_steps)) * N

    kappa = np.exp(jump_mean + 0.5 * jump_vol**2) - 1.0
    drift = (r - 0.5 * sigma**2 - jump_intensity * kappa) * dt

    log_returns = drift + sigma * np.sqrt(dt) * Z + J

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = np.log(S0)
    log_paths[:, 1:] = np.log(S0) + np.cumsum(log_returns, axis=1)

    return np.exp(log_paths)


jump_paths = simulate_jump_diffusion_paths(
    config.n_test_paths,
    config.n_steps,
    config.S0,
    config.r,
    config.sigma_train,
    config.T,
    jump_intensity=1.0,
    jump_mean=-0.08,
    jump_vol=0.12,
    seed=config.seed + 777
)

jump_delta_pos = bsm_delta_positions(jump_paths, config.K, config.r, config.sigma_train, config.T)
jump_delta_eval = evaluate_hedge_numpy(jump_paths, jump_delta_pos, premium, config.K, config.transaction_cost_rate)

jump_neural_pos = neural_hedge_positions_numpy(jump_paths, policy, config)
if np.isfinite(jump_neural_pos).all():
    jump_neural_eval = evaluate_hedge_numpy(jump_paths, jump_neural_pos, premium, config.K, config.transaction_cost_rate)
else:
    jump_neural_eval = {
        "pnl": np.full(config.n_test_paths, np.nan),
        "transaction_cost": np.full(config.n_test_paths, np.nan),
        "turnover": np.full(config.n_test_paths, np.nan)
    }

jump_stress_report = pd.DataFrame([
    {"scenario": "jump_diffusion", "strategy": "bsm_delta", **pnl_risk_metrics(jump_delta_eval["pnl"], jump_delta_eval["transaction_cost"], jump_delta_eval["turnover"])},
    {"scenario": "jump_diffusion", "strategy": "deep_hedge_pytorch", **pnl_risk_metrics(jump_neural_eval["pnl"], jump_neural_eval["transaction_cost"], jump_neural_eval["turnover"])}
])

jump_stress_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(jump_delta_eval["pnl"], bins=80, alpha=0.5, density=True, label="BSM delta")
if np.isfinite(jump_neural_eval["pnl"]).any():
    plt.hist(jump_neural_eval["pnl"], bins=80, alpha=0.5, density=True, label="Deep hedge")
plt.axvline(0, linestyle="--")
plt.title("Jump-Diffusion Stress PnL")
plt.xlabel("PnL")
plt.ylabel("Density")
plt.legend()
plt.show()

## 19. Transaction cost sensitivity

Cost level changes the optimal hedging behaviour.

A high-cost environment should favour less frequent or less aggressive trading.

We evaluate fixed trained policies under several cost rates.

For a full study, one should retrain the neural policy for each cost regime.

In [ ]:
cost_grid = [0.0, 0.00025, 0.00050, 0.00100, 0.00200, 0.00500]
cost_rows = []

for cost in cost_grid:
    delta_out = evaluate_hedge_numpy(test_paths, delta_hedge_positions, premium, config.K, cost)
    neural_out = evaluate_hedge_numpy(test_paths, neural_positions, premium, config.K, cost) if np.isfinite(neural_positions).all() else neural_eval

    cost_rows.append({"cost_rate": cost, "strategy": "bsm_delta", **pnl_risk_metrics(delta_out["pnl"], delta_out["transaction_cost"], delta_out["turnover"])})
    cost_rows.append({"cost_rate": cost, "strategy": "deep_hedge_pytorch", **pnl_risk_metrics(neural_out["pnl"], neural_out["transaction_cost"], neural_out["turnover"])})

cost_sensitivity = pd.DataFrame(cost_rows)

cost_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
for strategy, g in cost_sensitivity.groupby("strategy"):
    plt.plot(g["cost_rate"], g["loss_cvar_95"], marker="o", label=strategy)
plt.title("Cost Sensitivity: Loss CVaR 95%")
plt.xlabel("Transaction cost rate")
plt.ylabel("Loss CVaR 95%")
plt.legend()
plt.show()

## 20. PnL attribution

For each strategy, terminal PnL comes from:

$$
premium - payoff + hedge\_pnl - transaction\_cost
$$

We compare the components to understand whether a strategy is reducing risk or simply spending less on trading.

In [ ]:
def attribution_summary(label, evaluation):
    return {
        "strategy": label,
        "premium": premium,
        "mean_payoff": float(np.nanmean(evaluation["payoff"])),
        "mean_hedge_pnl": float(np.nanmean(evaluation["hedge_pnl"])),
        "mean_transaction_cost": float(np.nanmean(evaluation["transaction_cost"])),
        "mean_terminal_pnl": float(np.nanmean(evaluation["pnl"])),
        "std_terminal_pnl": float(np.nanstd(evaluation["pnl"], ddof=1))
    }


attribution = pd.DataFrame([
    attribution_summary("no_hedge", no_hedge_eval),
    attribution_summary("bsm_delta", delta_eval),
    attribution_summary("deep_hedge_pytorch", neural_eval)
])

attribution

In [ ]:
component_cols = ["mean_payoff", "mean_hedge_pnl", "mean_transaction_cost", "mean_terminal_pnl"]
plot_attr = attribution.set_index("strategy")[component_cols]

plt.figure(figsize=(12, 6))
x = np.arange(len(plot_attr.index))
width = 0.2

for i, col in enumerate(component_cols):
    plt.bar(x + (i - 1.5) * width, plot_attr[col], width=width, label=col)

plt.xticks(x, plot_attr.index, rotation=20)
plt.axhline(0, linestyle="--")
plt.title("PnL Attribution")
plt.ylabel("Mean amount")
plt.legend()
plt.tight_layout()
plt.show()

## 21. Model card

A responsible deep hedging notebook should report:

| Item | Status |
|---|---|
| Simulator | GBM training paths |
| Product | European call liability |
| Hedge instrument | underlying only |
| Cost model | proportional transaction cost |
| Objective | mean-variance terminal PnL |
| Baselines | no hedge, Black-Scholes delta |
| Stress tests | volatility mismatch and jumps |
| Claim of real-market superiority | not made |

This is risk-learning infrastructure, not a production hedge engine.

In [ ]:
model_card = pd.Series({
    "notebook": "03_16_deep_hedging_intro_pytorch",
    "torch_available": TORCH_AVAILABLE,
    "product": "European call short liability",
    "hedge_instrument": "underlying asset",
    "training_simulator": "GBM",
    "training_sigma": config.sigma_train,
    "transaction_cost_rate": config.transaction_cost_rate,
    "objective": "mean_variance_terminal_pnl",
    "baselines": "no hedge; Black-Scholes delta hedge",
    "stress_tests": "volatility mismatch; jump diffusion",
    "real_market_superiority_claim": "not made"
})

model_card

## 22. Practical checklist for deep hedging

Before trusting a learned hedge policy, check:

1. **Simulator realism**  
   Does the simulator reflect the market risk you care about?

2. **Baselines**  
   Does the network beat no hedge and classical delta baselines?

3. **Objective alignment**  
   Is the loss function aligned with desk risk preferences?

4. **Transaction costs**  
   Are costs realistic and included during training?

5. **Out-of-sample validation**  
   Are test paths independent of training paths?

6. **Stress testing**  
   What happens under wrong volatility, jumps, and heavy tails?

7. **Turnover**  
   Does the network reduce risk by excessive trading?

8. **Constraints**  
   Are hedge limits and liquidity constraints included?

9. **Interpretability**  
   Does hedge behaviour make economic sense?

10. **Model risk**  
   Does performance collapse when assumptions change?

11. **Reproducibility**  
   Are seeds, simulator parameters, and model weights tracked?

12. **Governance**  
   Would this pass a model validation review?

## 23. Saving outputs

The notebook saves:

1. configuration;
2. training history;
3. baseline strategy metrics;
4. neural hedge metrics;
5. hedge positions;
6. volatility stress report;
7. jump stress report;
8. cost sensitivity;
9. PnL attribution;
10. model card;
11. manifest.

In [ ]:
output_dir = Path("data/processed/deep_hedging_intro_pytorch_v1")
output_dir.mkdir(parents=True, exist_ok=True)

training_history_path = output_dir / "training_history.csv"
strategy_metrics_path = output_dir / "strategy_metrics.csv"
delta_positions_path = output_dir / "bsm_delta_positions_sample.csv"
neural_positions_path = output_dir / "neural_positions_sample.csv"
vol_stress_path = output_dir / "volatility_stress_report.csv"
jump_stress_path = output_dir / "jump_stress_report.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
attribution_path = output_dir / "pnl_attribution.csv"
model_card_path = output_dir / "model_card.csv"
config_path = output_dir / "config.json"
manifest_path = output_dir / "manifest.json"

training_history.to_csv(training_history_path, index=False)
strategy_metrics.to_csv(strategy_metrics_path, index=False)
pd.DataFrame(delta_hedge_positions[:100]).to_csv(delta_positions_path, index=False)
pd.DataFrame(neural_positions[:100]).to_csv(neural_positions_path, index=False)
vol_stress_report.to_csv(vol_stress_path, index=False)
jump_stress_report.to_csv(jump_stress_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
attribution.to_csv(attribution_path, index=False)
model_card.to_frame("value").to_csv(model_card_path)
Path(config_path).write_text(json.dumps(asdict(config), indent=2, default=str))

manifest = {
    "dataset_name": "deep_hedging_intro_pytorch_outputs",
    "schema_version": "deep_hedging_intro_pytorch_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "torch_available": TORCH_AVAILABLE,
    "initial_call_price": initial_call_price,
    "initial_call_delta": initial_call_delta,
    "strategy_metrics": strategy_metrics.to_dict(orient="records"),
    "model_card": model_card.to_dict(),
    "notes": [
        "Notebook introduces deep hedging as a learned risk-control policy, not market prediction.",
        "Policy is trained on GBM simulated paths if PyTorch is available.",
        "Baselines include no hedge and Black-Scholes delta hedge.",
        "Terminal PnL includes option premium, payoff, hedge PnL, and proportional transaction costs.",
        "Stress tests include volatility mismatch and jump diffusion.",
        "This is an introductory research notebook, not a production hedging engine."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

training_history_path, strategy_metrics_path, vol_stress_path, model_card_path, manifest_path

## 24. Limitations

### 24.1 Simulator dependence

The neural policy learns from the simulator.

If the simulator is unrealistic, the hedge may be unrealistic.

### 24.2 GBM training environment

GBM assumes continuous paths, constant volatility, and lognormal returns.

Real markets have stochastic volatility, jumps, liquidity effects, and regime changes.

### 24.3 Simple product

The notebook hedges one European call.

Real portfolios contain many instruments, maturities, strikes, path dependencies, and Greeks.

### 24.4 Simple cost model

Costs are proportional to notional turnover.

Real costs include bid/ask spread, market impact, latency, fees, funding, and liquidity constraints.

### 24.5 No inventory constraints beyond hedge bound

Production hedging may require position limits, risk limits, and margin constraints.

### 24.6 Basic neural architecture

The policy is a small MLP.

More advanced models may use recurrent networks, attention, or path-level state variables.

### 24.7 No formal model validation

A production deep hedging model would require extensive governance, explainability, stress testing, and independent validation.

## 25. Research frontier and extensions

Important extensions include:

1. **Train under stochastic volatility**  
   Heston, SABR, rough volatility, or local volatility simulators.

2. **Jump-aware deep hedging**  
   Train with jump diffusion or historical bootstrapped paths.

3. **Multi-option portfolio hedging**  
   Hedge a book rather than one option.

4. **CVaR and utility objectives**  
   Optimise explicit tail-risk or utility losses.

5. **Recurrent hedge policy**  
   Add memory of previous path history.

6. **Market-impact model**  
   Penalise large trades nonlinearly.

7. **Delta-vega hedging**  
   Include options as hedge instruments.

8. **Adversarial stress training**  
   Train under worst-case parameter perturbations.

9. **Deep hedging with real historical paths**  
   Bootstrap or generative market simulators.

10. **Chinese futures options extension**  
   Use Black-76 futures options, night sessions, price limits, and commodity-specific volatility regimes.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_17_feature_importance_and_shap_for_alpha**  
   Interpret learned models.

2. **04_07_tail_risk_cvar_optimization**  
   Study CVaR objectives more formally.

3. **05_04_option_strategy_backtester_delta_hedged**  
   Build realistic delta-hedged option PnL.

4. **06_12_gpu_monte_carlo_engine**  
   Scale simulation and neural training.

5. **07_02_volatility_surface_pricing_and_alpha**  
   Combine surface modelling, hedging, and option strategy research.

6. **Deep hedging capstone**  
   Train hedge policies under stochastic volatility, jumps, and transaction costs.

## 27. Summary

This notebook introduced deep hedging with PyTorch.

It showed how to:

1. simulate GBM training and test paths;
2. price a European call using Black-Scholes;
3. compute Black-Scholes delta baselines;
4. implement discrete hedging PnL with transaction costs;
5. define terminal PnL risk metrics;
6. build a neural hedge policy;
7. train the policy with a differentiable hedging loss;
8. compare no hedge, delta hedge, and neural hedge;
9. visualise hedge behaviour;
10. stress test volatility mismatch;
11. stress test jump diffusion;
12. test transaction-cost sensitivity;
13. attribute PnL components;
14. save outputs and metadata.

The key computational takeaway:

> Deep hedging turns hedging into a differentiable policy optimisation problem.

The key financial takeaway:

> A learned hedge is only as credible as its simulator, objective, frictions, baselines, and stress tests.

## 28. Further reading

- Buehler et al., "Deep Hedging."
- Black and Scholes, "The Pricing of Options and Corporate Liabilities."
- Merton, "Theory of Rational Option Pricing."
- Ruf and Wang on neural networks for option pricing and hedging.
- Literature on utility-based hedging, transaction costs, CVaR optimisation, and reinforcement learning for hedging.